In [0]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS turismo;
CREATE VOLUME IF NOT EXISTS workspace.turismo.dataset_turismo;

In [0]:
import os

workspace_path = "/Workspace/Users/airtonlirajr@gmail.com/rag_end_to_end_databricks/ingestion/kaggle.json"

# Diretório local permitido (mapeado ao Workspace)
conf_dir = "/Workspace/kaggle_conf"

os.makedirs(conf_dir, exist_ok=True)

# Agora pode copiar: /Workspace é o único prefixo local aceito pelo dbutils.fs
dbutils.fs.cp(workspace_path, f"file:{conf_dir}/kaggle.json", recurse=False)

os.system("chmod 600 /Workspace/kaggle_conf/kaggle.json")

# Aponta o Kaggle para esse diretório
os.environ["KAGGLE_CONFIG_DIR"] = conf_dir

In [0]:
print(os.listdir(conf_dir))
print(os.getenv("KAGGLE_CONFIG_DIR"))
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()
print("Autenticado com Kaggle!")

In [0]:
import kagglehub
from kagglehub import dataset_load, KaggleDatasetAdapter

datasets_names = []

datasets_names.append("aqua-rio-rj.csv")
datasets_names.append("beto-carreiro-sc.csv")
datasets_names.append("elevador_lacerda.csv")
datasets_names.append("hopi_hari.csv")
datasets_names.append("jardim_botanico.csv")
datasets_names.append("mercado-central-fortaleza-ce.csv")
datasets_names.append("mercado-ver-o-peso-pa.csv")
datasets_names.append("museu_arte_sp.csv")
datasets_names.append("museu_imperial.csv")
datasets_names.append("parque_chapada_dos_veadeiros.csv")
datasets_names.append("parque_jalapao.csv")
datasets_names.append("parque_nacional_iguacu.csv")
datasets_names.append("pelourinho_ba.csv")
datasets_names.append("praca-3-poderes-br.csv")
datasets_names.append("praia-copacabana-rj.csv")

#https://www.kaggle.com/datasets/jeffersonsfilho/dadosatracoesturisticasbr


for dataset_name in datasets_names:
    print(f"Carregando dataset: {dataset_name} .....")
    df = dataset_load(
        adapter=KaggleDatasetAdapter.PANDAS,
        handle="jeffersonsfilho/dadosatracoesturisticasbr",
        path=dataset_name,
        pandas_kwargs={"sep": ";"}
    )
    print(df.shape)
    df.head()

    print(f"Dataset: {dataset_name} carregado com sucesso!....")

    VOLUME_PATH = f"/Volumes/workspace/turismo/dataset_turismo/{dataset_name}"

    # Converte para Spark DF (inferSchema automático)
    spark_df = spark.createDataFrame(df)
    spark_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(VOLUME_PATH)

    spark.sql(f"""
        CREATE OR REPLACE TABLE workspace.turismo.bronze_{dataset_name.replace(".csv", "").replace("-","_")}
        USING DELTA
        AS SELECT * FROM read_files('{VOLUME_PATH}',
        format => 'csv',
        header => true,
        inferSchema => true
        )
    """)
    print(f"Dataset: {dataset_name} gravado no delta com sucesso!....")
